# TimeKAN layer-depth study: native R on Kaggle
Enable **GPU T4 x2**, **Internet on** and **Files only** persistence.
This runs 21 single-seed experiments on ETTh1, ETTh2 and ETTm1.
L = 1, 2, 3, 4, 5, 10, 20; seed 0; all other settings fixed.
Python installs/orchestrates; the models, data processing, training, evaluation and plots are R.
Read `docs/DEPTH_EXPERIMENT.md` before interpreting a plateau.


In [ ]:
from pathlib import Path
import os, subprocess, shutil, sys
print(subprocess.check_output(['nvidia-smi'],text=True))
assert shutil.which('Rscript'), 'An R installation is required.'
repo=Path('/kaggle/working/atsa-timekan')
if not repo.exists():
    subprocess.run(['git','clone','https://github.com/rviz190606/atsa-timekan.git',str(repo)],check=True)
os.chdir(repo)
assert (repo/'run_depth_experiment.R').exists(), 'Publish the depth-study code to main first.'
os.environ['R_LIBS_USER']='/kaggle/working/R-library'
def stream(cmd):
    p=subprocess.Popen(cmd,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
    for line in p.stdout: print(line,end='',flush=True)
    if p.wait(): raise RuntimeError('Command failed: '+str(cmd))
print('Commit:',subprocess.check_output(['git','rev-parse','HEAD'],text=True).strip())


In [ ]:
# Official CUDA R torch binaries; persistent library and resumable installer cache.
# The install helper emits download progress. A ready library is reused.
stream([sys.executable,'-u','notebooks/setup_kaggle_depth.py'])
stream(['Rscript','run_depth_experiment.R','--task','self_test'])


In [ ]:
# Only this depth sweep runs. Completed jobs are reused after an interruption.
stream([sys.executable,'-u','run_depth_kaggle.py'])


In [ ]:
import pandas as pd, base64
from IPython.display import display, Image, HTML
table=pd.read_csv(repo/'results/depth/summary.csv')
assert len(table)==21 and not table.duplicated(['dataset','layers','seed']).any()
assert set(table.seed)=={0}
display(table[['dataset','layers','test_mse','test_mae','params','minutes','best_epoch','hit_epoch_limit']])
for name in ['depth_accuracy_cost.png','depth_validation_curves.png']:
    display(Image(filename=str(repo/'figures/depth'/name)))
bundle=repo/'TimeKAN_depth_results.zip'
encoded=base64.b64encode(bundle.read_bytes()).decode()
display(HTML('<a download="TimeKAN_depth_results.zip" href="data:application/zip;base64,'+encoded+'">Download complete TimeKAN depth results</a>'))
print('VERIFIED: 21 runs, seed 0 only')
